# soundtouch-zonemaster Quickstart

`soundtouch-zonemaster` is a software zone master for Bose SoundTouch speakers: it lets several
speakers play one stream together after Bose shut down the cloud service that used to do this.
The package ships two command-line programs:

* `soundtouch-zonemaster` - the prototype: holds a zone against real speakers for a measured run.
* `soundtouch-zonemaster-service` - the service: reads its settings from layered configuration
  files and holds a house's zone until it is stopped.

Neither speaks to a real speaker in this notebook - everything below is safe to run with no
speakers on the network and produces no sound.

## 1) Install the package

Installed non-editable from the checkout this notebook runs in, so a fresh kernel (no prior
`pip install -e`) can still import it.

In [ ]:
import os
import sys
from pathlib import Path

!{sys.executable} -m pip install --quiet .
assert _exit_code == 0

# The console scripts pip just installed land beside sys.executable, which is not necessarily
# on PATH for the shell `!` spawns (a fresh venv, unactivated) - put that directory first so the
# commands below resolve.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")

## 2) CLI: version

Both console scripts print their name and version and exit; neither touches a speaker.

In [ ]:
!soundtouch-zonemaster --version
assert _exit_code == 0

!soundtouch-zonemaster-service --version
assert _exit_code == 0

## 3) CLI: read one configuration scope

Every CLI here speaks JSON: `--json-bare` prints one line, easy to parse. `config --section dialling`
reports the shipped defaults for the dialling scope (how long digits are collected into one
number, and how long a key must be held to count as a hold) and where each value came from. It
reads no network and starts nothing.

In [ ]:
import json
import tempfile
from pathlib import Path

config_path = Path(tempfile.gettempdir()) / "zonemaster-quickstart-config.json"

!soundtouch-zonemaster-service --json-bare config --section dialling > {config_path}
assert _exit_code == 0

with config_path.open(encoding="utf-8") as handle:
    envelope = json.load(handle)

assert envelope["ok"] is True
envelope

## 4) A pure domain example: directory play order

The `domain` layer holds no I/O and no framework, so it can be imported and exercised directly.
`play_order` is the rule a directory channel plays its files in: the files of one level first,
then its subdirectories, each played to its end before the next one begins, with names compared
naturally so chapter 2 sorts before chapter 10.

In [ ]:
from soundtouch_zonemaster.domain.playorder import play_order

files = (
    "Album/10 Track Ten.mp3",
    "Album/2 Track Two.mp3",
    "Album/Bonus/1 Extra.mp3",
    "Album/1 Track One.mp3",
)

play_order(files)